# Vérifier la connexion à PostgreSQL

Exécutez la cellule suivante dans **JupyterLab de la sandbox Docker** (Maj + Entrée).
Elle vérifie l'authentification, le nom de la base, l'utilisateur et le résultat d'une requête SQL.
Aucune table n'est créée ou modifiée. Vous pouvez relancer le test.

La configuration est fournie par Docker : `PGHOST=postgres`, `PGPORT=5432`,
`PGDATABASE=formation`, `PGUSER=apprenant`. Le mot de passe provient de `PGPASSWORD` ; il n'est pas affiché.
`postgres` est le nom du service Docker : `localhost` désignerait le conteneur Jupyter.

La requête exécutée par `scripts/test_postgres.py` est :

```sql
SELECT current_database() AS base, current_user AS utilisateur,
       version() AS version, 1 + 1 AS resultat;
```


In [ ]:
from pathlib import Path
import runpy

# Fonctionne depuis la racine de la sandbox ou depuis notebooks/.
base = Path.cwd()
if not (base / "scripts" / "test_postgres.py").is_file():
    base = base.parent
script = base / "scripts" / "test_postgres.py"
if not script.is_file():
    raise FileNotFoundError("Conservez scripts/ et notebooks/ dans la même sandbox.")

test = runpy.run_path(str(script))
resultat = test["verifier_connexion"]()


## Résultat attendu

```text
Connexion PostgreSQL OK
Serveur : postgres:5432
Base : formation | Utilisateur : apprenant
Test SQL : 1 + 1 = 2
PostgreSQL 17...
```

La base et l'utilisateur reflètent vos valeurs si vous avez personnalisé `.env`.
Un message « OK » signifie que la connexion et la requête ont réellement réussi.

## En cas d'échec

Dans un terminal, depuis la sandbox :

```bash
docker compose up --build -d
docker compose ps
docker compose logs postgres
```

- `ModuleNotFoundError: psycopg` : reconstruire le service notebook avec la commande ci-dessus.
- `postgres` doit être indiqué `healthy` ; vérifier ses logs sinon.
- Après modification des variables, recréer notebook avec `docker compose up -d --force-recreate notebook`, puis rouvrir ce notebook.
- Les identifiants définis à la première initialisation restent dans le volume PostgreSQL. Changer `.env` ensuite ne modifie pas les utilisateurs déjà créés.

Le test vérifie l'accès SQL, pas le chargement des données commerciales. Les exercices Matplotlib continuent à lire leur CSV.
